In [1]:
import os
import glob
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm

In [2]:
def read_booknlp(path_book):
    """Load the .book file, return a list of dicts (top N=10 characters)."""
    with open(path_book, "r", encoding="utf-8") as file:
        lines = file.readlines()
        # Evaluate each JSON-like line into a dictionary.
        dicts = [eval(line.strip()) for line in lines if line.strip()]
    return dicts[:10]  # keep only the first 10 characters


def get_characterization_all(booknlp_data):
    """
    Return a list of lists.
    Each element of the outer list is all the tokens (agent/patient/mod) for one character.
    """
    list_tout_mots = []
    for perso in booknlp_data:
        list_mots_par_perso = []
        list_mots_par_perso.extend([item['w'] for item in perso.get('agent', [])])
        list_mots_par_perso.extend([item['w'] for item in perso.get('patient', [])])
        list_mots_par_perso.extend([item['w'] for item in perso.get('mod', [])])
        list_mots_par_perso.extend([item['w'] for item in perso.get('poss', [])])
        list_tout_mots.append(list_mots_par_perso)
    return list_tout_mots

In [3]:
############################################################
# 3) GET ALL BOOK FILES AND BUILD A MASTER WORD LIST
############################################################

# Define the folder where your .book files are located (adjust as needed)
books_folder = "/home/crazyjeannot/OUTPUT_BOOK_GEN_Z/"
#books_folder = "/home/crazyjeannot/OUTPUT_BOOK_GEN_Z/BOOK_FILES/"
book_files = glob.glob(os.path.join(books_folder, "*.book"))

all_characters_words = []

for filepath in tqdm(book_files[:300]):#10% du corpus
    book_data = read_booknlp(filepath)
    characters_words = get_characterization_all(book_data)
    for list_word in characters_words:
        all_characters_words.extend(list_word)

# Now choose the top 1000 words in the entire corpus (instead of 100 for small example)
N_MFW = 10000
counter_characters = Counter(all_characters_words)
most_common_items = counter_characters.most_common(N_MFW)
top_n_words = [w for (w, count) in most_common_items]

print("Total tokens extracted for all characters:", len(all_characters_words))
print("Top 10 MFW for reference:", top_n_words[:10])
print("BOTTOM 10 FW for reference:", top_n_words[-10:])

  0%|          | 0/300 [00:00<?, ?it/s]

Total tokens extracted for all characters: 1581983
Top 10 MFW for reference: ['avoir', 'dire', 'voir', 'faire', 'vouloir', 'pouvoir', 'aller', 'savoir', 'aimer', 'jeune']
BOTTOM 10 FW for reference: ['glabre', 'fente', 'hublot', 'diffus', 'influençable', 'paratonnerre', 'rasage', 'tabou', 'flipper', 'vieillissement']


In [4]:
# ------------------------------
# 1) Helper Functions
# ------------------------------

def get_characterization(char_data):
    """
    Extract tokens for a character from various fields: agent, patient, mod, poss.
    Returns a list of tokens.
    """
    tokens = []
    tokens.extend([item['w'] for item in char_data.get('agent', [])])
    tokens.extend([item['w'] for item in char_data.get('patient', [])])
    tokens.extend([item['w'] for item in char_data.get('mod', [])])
    tokens.extend([item['w'] for item in char_data.get('poss', [])])
    return tokens

def get_character_name(char_dict):
    """
    Retrieve a character name from the 'mentions' field.
    If no proper mention is available, returns a fallback name using the character id.
    """
    if "mentions" in char_dict:
        proper = char_dict["mentions"].get("proper", [])
        if len(proper) > 0:
            # Use the first proper mention (which is usually the most frequent)
            return proper[0]["n"].lower()
    return f"char{char_dict.get('id', 'unknown')}"

In [ ]:
# ------------------------------
# 2) Build df_chapitres from All .book Files in a Folder
# ------------------------------

data_rows = []

# Loop over every .book file
for file_path in tqdm(book_files):
    base_name = os.path.basename(file_path).replace(".book", "")
    # Read the first 10 characters from the file
    book_data = read_booknlp(file_path)
    
    # Process each character dictionary
    for char_dict in book_data:
        char_id = char_dict.get("id", None)
        char_name = get_character_name(char_dict)
        tokens = get_characterization(char_dict)
        total_tokens = len(tokens)
        counter_tokens = Counter(tokens)
        
        # Prepare a row dictionary with metadata.
        row_data = {
            "text_id": base_name,        # file (text) id
            "char_id": char_id,          # character id from BookNLP
            "char_name": char_name,      # derived character name
            "total_tokens": total_tokens # optional: total token count for that character
        }
        # Add features: normalized frequency for each word in top_n_words.
        for w in top_n_words:
            row_data[w] = counter_tokens.get(w, 0) / total_tokens if total_tokens > 0 else 0.0
        
        data_rows.append(row_data)

# Build the output dataframe from all rows
df_chapitres = pd.DataFrame(data_rows)
print("df_chapitres shape:", df_chapitres.shape)

  0%|          | 0/2976 [00:00<?, ?it/s]

In [9]:
df_chapitres.to_csv("DF_BOW_ALL_CHAPITRES.csv", header=True, index=False)